# Day 4 — Feature Engineering & Hyperparameter Tuning


This notebook focuses on creating meaningful engineered features and tuning a Random Forest classifier using GridSearchCV with 5-fold cross-validation. The tuned model is compared with an untuned baseline, and the impact of engineered features and hyperparameters is analyzed.

## Step 1 — Setup and Data Loading

The required libraries are imported for data manipulation, preprocessing, model building, cross-validation, and hyperparameter tuning.

In [134]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories"
)


### Load the Dataset

The stroke dataset is loaded into a pandas DataFrame.

In [2]:
df= pd.read_csv("healthcare-dataset-stroke-data.csv")

### Inspect Dataset Columns

The column names are checked to understand the available features and identify the target and identifier columns.

In [3]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='str')

### Remove the ID Column

The `id` column is removed because it is only an identifier and does not provide meaningful predictive information for the model.

In [4]:
df = df.drop(columns="id")

## Feature Engineering

Two new features are created to provide the model with additional information derived from the existing variables.

- `age_group`: Groups patients into meaningful age categories using binning. This may help the model capture different patterns across age groups.
- `health_risk_count`: Counts selected health risk factors, including hypertension, heart disease, and smoking history.

These features were selected because they have a meaningful relationship with stroke risk rather than being arbitrary transformations.

The `age_group` feature divides age into four categories: Young, Adult, Middle-aged, and Senior.

In [5]:
df["age_group"]=pd.cut(df["age"],
                       bins=[0, 18, 40, 60,100],
                       labels=["Young","Adult","Middle_aged","Senior"])

The `health_risk_count` feature represents the number of selected risk factors for each patient.

A higher value indicates that the patient has more of the selected risk factors.

In [6]:
df["health_risk_count"] = (
    df["hypertension"]
    + df["heart_disease"]
    + df["smoking_status"].isin(
        ["smokes", "formerly smoked"]
    ).astype(int)
)

### Inspect the Engineered Features

The first few rows are displayed to verify that the new features were created correctly.

In [7]:
df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,age_group,health_risk_count
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,Senior,2
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1,Senior,0
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,Senior,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,Middle_aged,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,Senior,1


The new `age_group` and `health_risk_count` features were successfully added to the dataset.

### Separate Features and Target

The target variable `stroke` is separated from the input features.

In [9]:
X = df.drop(columns="stroke")
y = df["stroke"]

### Define Numerical and Categorical Features

The features are divided into numerical and categorical groups so that appropriate preprocessing can be applied to each type.

In [10]:
categorical_features = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status",
    "age_group"
]

numerical_features = [
    "age",
    "hypertension",
    "heart_disease",
    "avg_glucose_level",
    "bmi",
    "health_risk_count"
]

### Build the Preprocessing Pipeline

A `ColumnTransformer` is used to apply different preprocessing steps to numerical and categorical features.

- Missing numerical values are replaced using the median.
- Missing categorical values are replaced using the most frequent value.
- Categorical features are converted into numerical form using one-hot encoding.
- `handle_unknown="ignore"` prevents errors when unseen categories appear during cross-validation.

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

## Step 2 — Define the Hyperparameter Grid

A hyperparameter grid is defined for the Random Forest model.

The grid contains different values for the number of trees, maximum tree depth, minimum samples required to split a node, and minimum samples required at a leaf.

In [14]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

### Create the Random Forest Pipeline

The preprocessing steps and Random Forest model are combined into one pipeline.

`class_weight="balanced_subsample"` is used because the stroke dataset is highly imbalanced.

In [15]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model",  RandomForestClassifier(
    class_weight="balanced_subsample",
    random_state=42
))
])

### Define 5-Fold Stratified Cross-Validation

Stratified 5-fold cross-validation is used to evaluate the model while maintaining a similar proportion of stroke and non-stroke cases in each fold.

In [ ]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Step 3 — Run GridSearchCV

`GridSearchCV` evaluates every hyperparameter combination using 5-fold cross-validation.

The F1 score is used because the dataset is highly imbalanced and detecting the minority stroke class is more important than relying only on accuracy.

In [17]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=True
)

In [139]:
grid_search.fit(X, y);

### Best Hyperparameters

The best hyperparameter combination found by GridSearchCV is displayed below.

In [19]:
print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}


### Result
The best configuration used:

- `n_estimators = 200`
- `max_depth = 10`
- `min_samples_split = 5`
- `min_samples_leaf = 1`

### Best Cross-Validated Score

The best mean F1 score obtained during the 5-fold cross-validation is displayed.

In [20]:
print("Best Cross-Validated F1 Score:")
print(grid_search.best_score_)

Best Cross-Validated F1 Score:
0.20178235735404443


### Result
The best cross-validated F1 score was approximately **0.202**.

## Step 4 — Compare Tuned and Untuned Models

To measure the effect of hyperparameter tuning, an untuned Random Forest baseline is evaluated using the same 5-fold cross-validation strategy.

The baseline uses `n_estimators=200` while keeping the other Random Forest parameters at their default values, except for the class weighting and random state.

In [140]:
baseline_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced_subsample",
        random_state=42
    ))
])

The baseline model is evaluated using the same F1 scoring metric and 5-fold cross-validation.

In [141]:

baseline_scores = cross_val_score(
    baseline_rf,
    X,
    y,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

baseline_f1 = baseline_scores.mean()

print("Baseline CV F1:", baseline_f1)

Baseline CV F1: 0.015390307066222714


### Result
The untuned baseline achieved a mean 5-fold F1 score of approximately **0.0307**.

### Calculate the Improvement

The tuned model's cross-validated F1 score is compared with the untuned baseline.

In [132]:
tuned_f1 = grid_search.best_score_
improvement = tuned_f1 - baseline_f1

print("Tuned CV F1:", tuned_f1)
print("F1 Improvement:", improvement)

Tuned CV F1: 0.20178235735404443
F1 Improvement: 0.171001743221599


### Result
The tuned model achieved an F1 score of approximately **0.202**, compared with **0.031** for the baseline.

This represents an improvement of approximately **0.171** in the mean cross-validated F1 score.

This indicates that hyperparameter tuning substantially improved the Random Forest's performance on the selected metric.

## Step 5 — Hyperparameter Impact

After GridSearchCV tests all hyperparameter combinations, its results are stored in `cv_results_`.

We analyze these results to see which hyperparameter had the greatest effect on the mean cross-validated F1 score.

For each hyperparameter, we group the results by its tested values and calculate the average F1 score for each value. The value with the largest difference between its tested settings indicates a stronger impact on model performance.

In [142]:
# Store all GridSearchCV results in a DataFrame
results = pd.DataFrame(grid_search.cv_results_)

# List the hyperparameters we want to analyze
hyperparameters = [
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_split",
    "param_model__min_samples_leaf"
]

# Analyze each hyperparameter separately
for param in hyperparameters:

    print(f"\n{param}")

    # Group results by the hyperparameter value
    # and calculate the average cross-validated F1 score
    scores = (
        results.groupby(param)["mean_test_score"]
        .mean()
        .sort_values(ascending=False)
    )

    print(scores)


param_model__n_estimators
param_model__n_estimators
200    0.106472
100    0.105112
Name: mean_test_score, dtype: float64

param_model__max_depth
param_model__max_depth
10    0.191789
20    0.063637
Name: mean_test_score, dtype: float64

param_model__min_samples_split
param_model__min_samples_split
5    0.115333
2    0.096252
Name: mean_test_score, dtype: float64

param_model__min_samples_leaf
param_model__min_samples_leaf
2    0.118822
1    0.092762
Name: mean_test_score, dtype: float64



### Result
Among the tested hyperparameters, `max_depth` showed the largest difference in mean cross-validated F1 score.

A `max_depth` of 10 achieved a much higher mean F1 score than a `max_depth` of 20, making it the most influential hyperparameter among those tested.

### Prepare Feature Sets for Comparison

To evaluate the engineered features separately, the original numerical and categorical features are defined.

The Random Forest configuration is kept fixed while different feature sets are evaluated.

In [111]:
base_numerical = [
    "age",
    "hypertension",
    "heart_disease",
    "avg_glucose_level",
    "bmi"
]

base_categorical = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

### Create a Feature Evaluation Function

A function is created to evaluate different feature sets using the same preprocessing steps, Random Forest configuration, and 5-fold cross-validation.

This keeps the comparison fair because only the selected features change between experiments.

In [153]:
def evaluate_features(X, base_numerical, base_categorical):

    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), base_numerical),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ))
        ]), base_categorical)
    ])

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced_subsample",
            random_state=42
        ))
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="f1"
    )

    return scores.mean()

### Evaluate the Original Features

First, the Random Forest is evaluated using only the original features without the engineered features.

In [154]:
X_original = df.drop(
    columns=["stroke", "age_group", "health_risk_count"]
)

original_f1 = evaluate_features(
    X_original,
    base_numerical,
    base_categorical
)

### Evaluate `age_group`

The `age_group` feature is added to the original feature set while keeping the model configuration unchanged.

In [155]:
X_age = df.drop(
    columns=["stroke", "health_risk_count"]
)

age_f1 = evaluate_features(
    X_age,
    base_numerical,
    base_categorical + ["age_group"]
)

### Evaluate `health_risk_count`

The `health_risk_count` feature is added to the original feature set while keeping the same Random Forest configuration.

In [156]:
X_risk = df.drop(
    columns=["stroke", "age_group"]
)

risk_f1 = evaluate_features(
    X_risk,
    base_numerical + ["health_risk_count"],
    base_categorical
)

### Compare Engineered Features

The F1 scores from the three feature sets are compared to determine which engineered feature had the largest impact.

In [157]:
print("Feature Engineering Comparison")
print("------------------------------")
print("Original Features:", original_f1)
print("Original + age_group:", age_f1)
print("Original + health_risk_count:", risk_f1)

Feature Engineering Comparison
------------------------------
Original Features: 0.00784313725490196
Original + age_group: 0.007547169811320755
Original + health_risk_count: 0.0


## Feature and Hyperparameter Impact

The feature engineering comparison showed that neither engineered feature improved the mean 5-fold cross-validated F1 score compared with the original features.

The original features achieved an F1 score of **0.0078**, while adding `age_group` resulted in a slightly lower score of **0.0075**. Adding `health_risk_count` resulted in an F1 score of **0.0000**.

Therefore, neither engineered feature provided a positive improvement in F1 score in this experiment.

For the hyperparameters, `max_depth` had the largest impact among the tested parameters. A `max_depth` of **10** achieved a much higher mean cross-validated F1 score than `max_depth` of **20**.

Overall, the feature engineering experiments did not show a positive improvement in F1, while `max_depth` was the most influential hyperparameter among those tested.